# 深度Agent
---
[deepagents](https://docs.langchain.com/oss/python/deepagents/overview) 可以看作是 LangChain 团队出品的 DeepResearch. 本节通过查询一个近期新闻，检验 Deep Agents 是否有主动调用搜索，获取外部信息的能力。

In [1]:
# !pip install deepagents ddgs

In [2]:
import deepagents
import ddgs

print(deepagents.__version__)
print(ddgs.__version__)

0.4.3
9.6.1


**1）加载模型**

In [3]:
import os
from datetime import datetime
from dotenv import load_dotenv
from ddgs import DDGS
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from deepagents import create_deep_agent

# 加载模型配置
_ = load_dotenv()

# 加载模型
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model="qwen3-coder-plus",
    temperature=0.7,
)

**2）创建搜索工具**

[DDGS](https://github.com/deedy5/ddgs) 是一款免费的、无需 `API_KEY` 即可直接使用的网络搜索服务。可以作为`Tavily`懒人平替。

In [4]:
# 创建 ddgs 客户端
ddgs = DDGS()

def internet_search(
    query: str,
    max_results: int = 3,
) -> str:
    """
    使用互联网搜索指定关键词并返回格式化结果。

    Args:
        query: 搜索关键词或问题。
        max_results: 返回的最大结果条数。

    Returns:
        包含每条搜索结果的标题、摘要与链接的字符串。
    """
    results = list(
        ddgs.text(
            query=query,
            region="cn-zh",  # wt-wt zh
            timelimit='y',
            safesearch='off',  # moderate off
            page=1,
            backend='auto',
            max_results=max_results,
        )
    )

    content = ""
    for i, r in enumerate(results, 1):
        content += f"【结果 {i}】\n"
        content += f"标题: {r['title']}\n"
        content += f"摘要: {r['body']}\n"
        content += f"链接: {r['href']}\n\n"

    return content

In [5]:
# 测试
res = internet_search(query="openclaw 实践")
print(res)

【结果 1】
标题: OpenClaw
摘要: OpenClaw是一个开源的自主人工智能（AI）虛擬助理软件项目。其由软件工程师彼得·斯坦伯格（Peter Steinberger）开发，最初于2025年末以Clawdbot的名字在GitHub上发布，后更名为Moltbot，最终定为现名。2026年初，该AI项目因能够根据用户指令在应用程序和在线服务中自主处理复杂任务而受到关注。因其图标是一只红色龙虾，故中文又俗称为“龙虾”。
链接: https://zh.wikipedia.org/wiki/OpenClaw

【结果 2】
标题: OpenClaw 中文案例精选（awesome-openclaw-zh） - GitHub
摘要: 中文 OpenClaw 实战案例库：168 个可复制使用场景，从5分钟上手到完全精通。案例覆盖自动化、内容创作、运营增长与低成本稳定运行。 - cogine-ai/awesome- openclaw -zh
链接: https://github.com/cogine-ai/awesome-openclaw-zh

【结果 3】
标题: 玩转OpenClaw｜安装OpenClaw Skills及实践指南 - Tencent Cloud
摘要: 6 days ago · How to Install and Practice OpenClaw Skills: A Step-by-Step Guide 1. What is OpenClaw ? OpenClaw is an open-source robotics simulation environment designed for claw-based robotic arm manipulation tas...
链接: https://www.tencentcloud.com/techpedia/139686




**3）创建深度代理**

In [7]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """你是一名资深研究员。你的工作是进行全面深入的研究，并撰写一份精炼的报告。

你可以使用互联网搜索工具作为获取信息的主要方式。

## `internet_search`

使用该工具对指定查询进行互联网搜索。你可以设置返回结果的最大数量。

对于该工具的 query 参数，每次最多输入 **2个** 关键词。且关键词之间必须用空格分开。若不遵守此条规定，工具将返回无意义内容。

正确用例：

- 南京
- 美食 四川

错误用例：

- 美食 四川 2026

注意，当关键词数量为2个的时候，必须将2个词中更重要的那个放在前面。
"""

@tool
def get_today_date() -> str:
    """获取今天的日期"""
    return datetime.now().strftime("%Y-%m-%d")

agent = create_deep_agent(
    model=llm,
    tools=[internet_search, get_today_date],
    system_prompt=research_instructions
)

**4）运行 Agent**

我们的问题是：

```
openclaw是什么，它有什么强大的特性？
```

因为openclaw是2026年年初开源出来的一个项目，这条信息肯定不在预训练数据里。所以可以用这个问题验证 Deep Agents 是否真的联网了。

In [8]:
result = agent.invoke({"messages": [
    {"role": "user", "content": "openclaw是什么，它有什么强大的特性？"}
]})

# 最终回复
print(result["messages"][-1].content)

# OpenCLAW 研究报告

## 什么是 OpenCLAW？

OpenCLAW（前身为 Clawdbot/Moltbot）是一个开源的、本地优先（Local-First）的 AI Agent 框架，由 Peter Steinberger 开发。不同于传统的聊天机器人如 ChatGPT，OpenCLAW 是一个能够接管实际任务的自主 AI 私人助理，可以运行在用户的个人电脑上（支持 Mac、Windows 或 Linux 系统）。

## 主要特性与功能

### 1. 本地运行能力
- **本地优先架构**：运行在用户自己的设备上，确保数据隐私和安全
- **跨平台支持**：兼容主流操作系统（Mac、Windows、Linux）

### 2. 实际任务执行
- **文件管理**：能够管理和操作本地文件系统
- **浏览器控制**：可以直接控制浏览器执行网页操作
- **消息发送**：具备发送各类消息的能力
- **真实任务处理**：不仅仅是对话，而是能执行真实的计算机任务

### 3. 自主 AI 代理框架
- **持续进化能力**：具备长期可靠工作的能力，而非仅限于单次对话
- **规则与记忆集成**：整合规则、记忆和工具链，形成可持续进化的数字分身
- **工具链集成**：能够连接各种外部工具和服务

### 4. 部署灵活性
- **Docker 支持**：提供 Docker 镜像便于容器化部署
- **服务器部署**：可以在 VPS 或服务器上长期运行
- **浏览器扩展集成**：通过 Chrome 扩展实现浏览器控制功能

### 5. 开源属性
- **开源框架**：代码公开，社区驱动开发
- **可定制性**：用户可以根据需求进行个性化定制
- **社区支持**：拥有活跃的开发者和用户社区

## 应用场景

OpenCLAW 适合需要执行实际任务的场景，如自动化办公、文件处理、网络操作等，是一个真正的 AI 助手而非单纯的聊天工具。其核心优势在于能够长期可靠地执行任务，构建个性化的数字分身，帮助用户完成日常的计算机操作任务。


In [9]:
# 思考过程
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

openclaw是什么，它有什么强大的特性？
================================== Ai Message ==================================

我来为您研究OpenCLAW的相关信息。

首先，让我通过网络搜索了解OpenCLAW是什么以及它的特性。
Tool Calls:
  internet_search (call_2d4bbb7b71af42cba26c4904)
 Call ID: call_2d4bbb7b71af42cba26c4904
  Args:
    query: OpenCLAW 是什么
  internet_search (call_07aaf2baa30a4b21ba487500)
 Call ID: call_07aaf2baa30a4b21ba487500
  Args:
    query: OpenCLAW 特性 功能
================================= Tool Message =================================
Name: internet_search

【结果 1】
标题: OpenClaw 是什么 ？一篇讲清它能做 什么 、不能做 什么
摘要: 2026年2月27日 · OpenClaw 真正有意思的地方，不在于它“会不会说”，而在于它“能不能长期可靠地做”。 当你把规则、记忆、工具链都接起来，它就不再是聊天机器人，而是一个可持续进化的数字分身。 …
链接: https://zhuanlan.zhihu.com/p/2010863326960316590

【结果 2】
标题: OpenClaw | 能干活的 AI 助手
摘要: 2026年3月2日 · OpenClaw 是 Peter Steinberger 开发的开源自主 AI 私人助理。 它运行在你自己的电脑上（Mac、Windows 或 Linux），能执行真实任务——管理文件、控制浏览器、发送消息，还能在 …
链接: https://openclaws.io/zh/

【